# A Vulnerability Prioritization and Exposure Management (VPEM) Graph Use Case

## Introduction

This notebook demonstrates how to create a **Vulnerability Prioritization and Exposure Management (VPEM)** graph in Neo4j using a combination of infrastructure and threat intelligence data. The VPEM graph helps organizations move beyond simple severity scores (like CVSS) to prioritize vulnerabilities based on their **real-world reachability** and **potential business impact**.

This is a classic use case for cybersecurity teams looking to evolve from reactive "patching everything" to a proactive, risk-based vulnerability management strategy.

### Data Sources

In our example, we integrate three distinct layers of data to build a holistic risk view:

1. **Organizational Asset Context:** We will load synthetic data representing **Applications, Libraries, Build Artifacts, and Cloud Infrastructure** (such as Compute Instances and S3 Buckets). This layer includes the critical "Blast Radius" relationships, such as IAM Policies and Network Interfaces.
2. **Vulnerability Intelligence (NVD):** We will ingest official CVE data from the [NVD database](https://nvd.nist.gov/), including CVSS scores, attack vectors, and CWE problem types, sourced from both CNA and ADP containers.
3. **Threat Intelligence (CISA KEV):** We will enrich our CVE nodes with the [CISA Known Exploited Vulnerabilities (KEV)](https://www.cisa.gov/known-exploited-vulnerabilities-catalog) catalog to flag vulnerabilities that are currently being leveraged by threat actors in the wild.

### The Graph Advantage

Traditional vulnerability management relies on flat lists and spreadsheets, which often lead to "alert fatigue" because they treat a critical vulnerability on an isolated internal server the same as one on an internet-facing gateway.

By using Neo4j, we can perform **Attack Path Analysis** to answer critical questions:

* **Reachability:** Is this vulnerable library actually running on a server accessible from the public internet?
* **Impact:** If this server is compromised, what "Crown Jewel" assets (like PII databases) does its Identity have permission to access?
* **Efficiency:** Which single library update will remediate the highest aggregate risk across our entire production environment?

### The Resulting Schema

With the data loaded, our graph schema will look like the following.

![VPEM Graph Schema](vpem-schema.png)

We include nodes for Applications, Libraries, Build Artifacts, Compute Instances, IAM Policies, and CVEs. The relationships capture deployment, usage, and vulnerability identification.

## Architecture

### Overview

In our VPEM solution, the system is built on a **Security Knowledge Graph** architecture. Unlike traditional relational databases that struggle with deep path traversal (e.g., finding a path from a CVE to an S3 bucket), Neo4j allows us to map the complex relationships between code, infrastructure, and identity in real-time.

### Data Sources & Integration

To build a comprehensive exposure map, the system would typically ingest data from three primary domains:

| Domain | Data Sources | Key Entities Extracted |
| --- | --- | --- |
| **External Threat Intel** | NVD (CVE), CISA KEV, EPSS | Vulnerabilities, Exploit Status, CWEs |
| **Software Supply Chain** | SBOMs, GitHub/GitLab, JFrog Artifactory | Repos, Libraries, Build Artifacts |
| **Cloud Infrastructure** | AWS/Azure/GCP APIs, Wiz, Prisma Cloud | Compute, S3, IAM Roles, Subnets, Endpoints |

### Typical Data Flow

The architecture would follow a standard **Extract, Load, Transform (ELT)** pattern tailored for graph structures:

1. **Ingestion (Python):** Specialized collectors fetch raw JSON data from NVD and CISA. Simultaneously, internal scanners export infrastructure and application manifests.
2. **Entity Resolution (Neo4j MERGE):** Data is streamed into Neo4j using `MERGE` logic. This ensures that if a vulnerability (e.g., Log4j) is found in multiple libraries, it is represented as a single node with multiple relationships, preventing data silos.
3. **Contextual Enrichment:** Once the base nodes are loaded, the system runs graph algorithms to calculate "reachability." It looks for paths between `Endpoint` nodes (the internet) and `ComputeInstance` nodes containing vulnerabilities.
4. **Prioritization Engine:** A final scoring pass is performed. The engine combines the technical severity (CVSS) with the business context (Asset Criticality) and the blast radius (IAM Permissions) to generate a **Contextual Risk Score**.

Depending on an organization's needs, this architecture can be extended with additional data sources, integration points, etc.

### System Integration

The output of this architecture is typically consumed by:

* **Security Operations (SIEM/SOAR):** To prioritize incoming alerts based on the "Blast Radius" of the affected asset.
* **Engineering Teams:** To receive curated "Top 10" lists of library updates that provide the maximum risk reduction for their specific repositories.
* **Compliance & Audit:** To visualize and report on the "Aging" of vulnerabilities on critical PII-bearing systems.


## The Implementation

### Loading NVD Data into Neo4j

We first implement the necessary code to load NVD CVE data into the Neo4j graph database. This includes parsing the JSON files, creating nodes for CVEs, and establishing relationships with other entities in the graph.

In this implementation, we assume you have cloned the NVD Git repository locally with:

```bash
git clone https://github.com/CVEProject/cvelistV5.git
```

You could also change this implementation to pull data directly from the NVD API if preferred - we picked the local file approach for simplicity and speed.

In [8]:
import json
import os
from pathlib import Path
from neo4j import GraphDatabase
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

# Connection details
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "password"))
DB = os.getenv("NEO4J_DB", "nvd")
NVD_REPO_PATH = os.getenv("NVD_REPO_PATH", "./cvelistV5/cves/")
KEV_URL = os.getenv("KEV_URL", "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json")

def create_constraints():
    queries = [
        "CREATE CONSTRAINT cve_id_unique IF NOT EXISTS FOR (c:CVE) REQUIRE c.id IS UNIQUE",
        "CREATE CONSTRAINT cwe_id_unique IF NOT EXISTS FOR (w:CWE) REQUIRE w.id IS UNIQUE",
        "CREATE CONSTRAINT vendor_name_unique IF NOT EXISTS FOR (v:Vendor) REQUIRE v.name IS UNIQUE",
        "CREATE CONSTRAINT product_name_unique IF NOT EXISTS FOR (p:Product) REQUIRE p.name IS UNIQUE"
    ]
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            for query in queries:
                try:
                    session.run(query)
                except Exception as e:
                    print(f"Failed to create constraint: {e}")

def load_cve_batch(tx, batch):
    query = """
    UNWIND $batch AS data
    WITH data, data.containers.cna AS cna, data.containers.adp AS adp_list
    
    // 1. Create/Update CVE Node
    MERGE (v:CVE {id: data.cveMetadata.cveId})
    SET v.state = data.cveMetadata.state,
        v.assigner = data.cveMetadata.assignerShortName,
        v.description = [d IN cna.descriptions WHERE d.lang = 'en'][0].value
    
    // 2. Metrics Logic (Consolidated CNA and ADP)
    // We keep 'cna' and 'adp_list' in scope for subsequent sections
    WITH v, cna, adp_list,
         coalesce(cna.metrics, []) + 
         reduce(acc = [], x IN coalesce(adp_list, []) | acc + coalesce(x.metrics, [])) AS all_metrics
    
    WITH v, cna, adp_list, [m IN all_metrics WHERE m.cvssV3_1 IS NOT NULL OR m.cvssV3_0 IS NOT NULL][0] AS metric
    WITH v, cna, adp_list, coalesce(metric.cvssV3_1, metric.cvssV3_0) AS cvss
    
    FOREACH (_ IN CASE WHEN cvss IS NOT NULL THEN [1] ELSE [] END |
        SET v.baseScore = cvss.baseScore,
            v.baseSeverity = cvss.baseSeverity,
            v.attackComplexity = cvss.attackComplexity,
            v.attackVector = cvss.attackVector,
            v.availabilityImpact = cvss.availabilityImpact,
            v.confidentialityImpact = cvss.confidentialityImpact,
            v.integrityImpact = cvss.integrityImpact,
            v.privilegesRequired = cvss.privilegesRequired,
            v.userInteraction = cvss.userInteraction
    )

    // 3. Vendors and Products (Consolidated CNA and ADP)
    // Combine 'affected' lists from CNA and all ADP entries
    WITH v, cna,
         coalesce(cna.affected, []) + 
         reduce(acc = [], x IN coalesce(adp_list, []) | acc + coalesce(x.affected, [])) AS all_affected
    
    UNWIND all_affected AS affected
    WITH v, cna, affected 
    WHERE affected.product IS NOT NULL AND affected.product <> 'n/a'
      AND affected.vendor IS NOT NULL AND affected.vendor <> 'n/a'
    
    MERGE (vend:Vendor {name: affected.vendor})
    MERGE (p:Product {name: affected.product})
    MERGE (vend)-[:PROVIDES]->(p)
    MERGE (v)-[:AFFECTS]->(p)

    // 4. Problem Types (CWEs)
    WITH v, cna
    UNWIND coalesce(cna.problemTypes, []) AS pt
    UNWIND coalesce(pt.descriptions, []) AS desc
    WITH v, desc WHERE (desc.cweId IS NOT NULL AND desc.cweId =~ 'CWE-\\\\d+') 
                 OR (desc.description IS NOT NULL AND desc.description =~ 'CWE-\\\\d+')
    MERGE (w:CWE {id: coalesce(desc.cweId, desc.description)})
    MERGE (v)-[:HAS_PROBLEM_TYPE]->(w)
    """
    tx.run(query, batch=batch)

def process_local_repo(years: list = None):
    base_path = Path(NVD_REPO_PATH)
    batch_size = 1000
    current_batch = []
    total_processed = 0

    # Pre-collect filtered files to get a total count
    print("Scanning directory for files...")
    all_files = list(base_path.rglob("CVE-*.json"))
    if years:
        all_files = [f for f in all_files if any(str(y) in f.parts for y in years)]
    
    total_files = len(all_files)

    # Process with a full progress bar
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            # Wrap the list in tqdm for a % completion bar
            for json_file in tqdm(all_files, desc="Ingesting CVEs", unit="cve"):
                try:
                    with open(json_file, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        if data.get("cveMetadata", {}).get("state") == "PUBLISHED":
                            current_batch.append(data)
                except Exception as e:
                    # Use tqdm.write instead of print to avoid breaking the bar
                    tqdm.write(f"Error in {json_file.name}: {e}")

                if len(current_batch) >= batch_size:
                    session.execute_write(load_cve_batch, current_batch)
                    total_processed += len(current_batch)
                    current_batch = []

            if current_batch:
                session.execute_write(load_cve_batch, current_batch)
                total_processed += len(current_batch)

We also implement a helper method to load the CISA KEV catalog to flag known exploited vulnerabilities. This will become important later when we prioritize vulnerabilities based on real-world exploit activity.

In [9]:
import requests

def update_cisa_kev(url=None):
    print("Fetching CISA KEV Catalog...")
    response = requests.get(url)
    if response.status_code != 200:
        print("Failed to fetch CISA KEV data.")
        return
    
    data = response.json()
    vulnerabilities = data.get("vulnerabilities", [])
    
    query = """
    UNWIND $batch AS item
    // Match the existing CVE in your DB
    MATCH (v:CVE {id: item.cveID})
    
    // Add CISA specific metadata
    SET v.kev_addedDate = item.dateAdded,
        v.kev_dueDate = item.dueDate,
        v.kev_reason = item.shortDescription,
        v.isKnownExploited = true
    
    // Create a reference node for the Catalog itself
    MERGE (cat:Catalog {name: 'CISA KEV'})
    SET cat.lastUpdated = $timestamp,
        cat.version = $version
    
    MERGE (v)-[:LISTED_IN]->(cat)
    """
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            session.run(query, 
                        batch=vulnerabilities, 
                        timestamp=data.get('dateReleased'),
                        version=data.get('catalogVersion'))
            
    print(f"Successfully linked {len(vulnerabilities)} exploited vulnerabilities to the CISA KEV.")

## Loading data into Neo4j

We now proceed to load the data into Neo4j using the defined architecture and data flow. This involves creating the necessary database constraints, parsing the data files, and executing the Cypher queries to populate the graph. In our case we load CVE data between 2000 and 2025 - you might choose to limit this range based on your organization's needs.

In [10]:
create_constraints()

In [11]:
process_local_repo([
    2000, 2001, 2002, 2003, 2004,
    2005, 2006, 2007, 2008, 2009,
    2010, 2011, 2012, 2013, 2014,
    2015, 2016, 2017, 2018, 2019,
    2020, 2021, 2022, 2023, 2024,
    2025
    ])

Scanning directory for files...


Ingesting CVEs:   0%|          | 0/323749 [00:00<?, ?cve/s]

With the main CVE data loaded, we can now enrich our graph with the CISA KEV catalog to flag known exploited vulnerabilities.

In [12]:
update_cisa_kev(KEV_URL)

Fetching CISA KEV Catalog...
Successfully linked 1484 exploited vulnerabilities to the CISA KEV.


## Creating Sample Infrastructure Data

We now create sample infrastructure data representing applications, libraries, build artifacts, and cloud resources. This synthetic data will help us demonstrate the VPEM graph's capabilities in a controlled environment. We have chosen two scenarios: one high-risk path with internet exposure and a low-risk internal application.

Depending on your environment, you would model your actual infrastructure differently, but the principles remain the same.

In [ ]:
def create_vpem_data():
    query = """
        // Setup Libraries
        MERGE (l1:Library {name: 'log4j-core', version: '2.14.1', language: 'Java'})
        MERGE (l2:Library {name: 'spring-web', version: '5.3.8', language: 'Java'})
        
        // Scenario A: The High-Risk Path (Internet -> App -> S3)
        MERGE (repoA:Repo {name: 'customer-api-prod', criticality: 'High'})
        MERGE (artA:BuildArtifact {id: 'art-prod-001', registry: 'jfrog-prod'})
        MERGE (appA:Application {name: 'CustomerFacingAPI', tier: 'P1'})
        MERGE (insA:ComputeInstance {id: 'i-0001', name: 'api-gateway-01', public_ip: '34.201.1.5'})
        MERGE (endA:Endpoint {url: 'api.acme.com'})
        MERGE (idA:Identity {name: 'service-account-prod-s3', arn: 'arn:aws:iam::123:role/S3Access'})
        MERGE (polA:IAMPolicy {name: 'DataLakeFullAccess'})
        MERGE (cloudA:CloudService {name: 'S3-Bucket', resource_name: 'acme-customer-pii-data'})
        MERGE (teamA:Team {name: 'Platform-Security'})

        // Establish High-Risk Relationships
        MERGE (l1)-[:DEPENDENCY_OF]->(artA)
        MERGE (artA)-[:BUILT_FROM]->(repoA)
        MERGE (artA)-[:RUNNING_AS]->(appA)
        MERGE (appA)-[:HOSTED_ON]->(insA)
        MERGE (endA)-[:RESOLVES_TO]->(insA)
        MERGE (insA)-[:RUNS_AS]->(idA)
        MERGE (appA)-[:AUTHENTICATES_VIA]->(idA)
        MERGE (idA)-[:ASSUMES]->(polA)
        MERGE (polA)-[:HAS_ACCESS_TO]->(cloudA)
        MERGE (teamA)-[:MANAGES]->(idA)

        // Scenario B: The Low-Risk Path (Internal / Isolated)
        MERGE (repoB:Repo {name: 'internal-parser', criticality: 'Low'})
        MERGE (artB:BuildArtifact {id: 'art-dev-999'})
        MERGE (appB:Application {name: 'LegacyParser', tier: 'P3'})
        MERGE (insB:ComputeInstance {id: 'i-0002', name: 'internal-worker-01'}) // Note: No Public IP
        
        // Establish Low-Risk Relationships
        MERGE (l1)-[:DEPENDENCY_OF]->(artB) // Both apps use the same vulnerable library
        MERGE (artB)-[:BUILT_FROM]->(repoB)
        MERGE (artB)-[:RUNNING_AS]->(appB)
        MERGE (appB)-[:HOSTED_ON]->(insB)

        // Connect to Existing NVD Data (Mapping via CVE id)
        WITH l1
        MATCH (v:CVE) 
        WHERE v.id IN ['CVE-2021-44228', 'CVE-2021-45046']
        MERGE (v)-[:IDENTIFIED_IN]->(l1)
        """
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            session.run(query)

    print("VPEM data creation complete.")

In [14]:
create_vpem_data()

VPEM data creation complete.
